In [ ]:
import os
import mne
from mne.preprocessing import ICA
import numpy as np
import matplotlib.pyplot as plt

data_dir = os.path.abspath("./datasets")
os.makedirs(data_dir, exist_ok=True)

# 每次运行强制更新 config
mne.set_config("MNE_DATA", data_dir)
mne.set_config("MNE_DATASETS_SAMPLE_PATH", data_dir)


print("config file =", mne.get_config_path())
# 打印config 内容,友好格式
config = mne.get_config()
for key, value in config.items():
    print(f"{key}: {value}")



sample_data_path = mne.datasets.sample.data_path(update_path=True)
raw_fname = sample_data_path / "MEG" / "sample" / "sample_audvis_raw.fif"


In [ ]:
# 下载/加载 Sample 数据集
raw = mne.io.read_raw_fif(raw_fname, preload=True,verbose="warning")

# 只看eeg通道
raw_eeg = raw.copy().pick("eeg")  # 这个对象里只剩 EEG

# 查看基本信息：通道数、采样率、时长、通道类型等
# print(raw.info)
print(f"采样率: {raw_eeg.info['sfreq']} Hz")
print(f"通道数: {raw_eeg.info['nchan']}")
print(f"数据时长: {raw_eeg.times[-1]:.1f} 秒")
print(f"初始坏导列表: {raw_eeg.info['bads']}")

In [ ]:
# ---- 取一个通道的数据 ----
sfreq = raw_eeg.info['sfreq']
data, times = raw_eeg[0]  # 第一个通道，shape (channel, n_times)
signal = data[0]          # 转一维数组

N = 1000

# 取前 1000 个样本
signal = signal[:N] * 1e6  # 转 μV
times = times[:N]

# ---- 手动 FFT ----
fft_vals = np.fft.rfft(signal) # 单边 FFT，没有x2
fft_freq = np.fft.rfftfreq(N, d=1.0 / sfreq)


# 振幅谱
amplitude = np.abs(fft_vals) / N
if N % 2 == 0:
    amplitude[1:-1] *= 2
else:
    amplitude[1:] *= 2

# 功率谱(实际周期图)
power = np.abs(fft_vals) ** 2 / N   # 功率谱(实际周期图):采集时间越长，y轴值越大,单位是 μV²
if N % 2 == 0:
    power[1:-1] *= 2
else:
    power[1:] *= 2

# 功率谱密度
df = sfreq / N
psd = (np.abs(fft_vals / N) ** 2) / df
if N % 2 == 0:
    psd[1:-1] *= 2
else:
    psd[1:] *= 2


# ---- 绘图 ----
fig, axes = plt.subplots(4, 1, figsize=(12, 8))

# 时域信号
axes[0].plot(times, signal, linewidth=0.5, color='steelblue')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude (μV)')
axes[0].set_title(f'EEG Signal - {raw_eeg.ch_names[0]}')

axes[1].plot(fft_freq, amplitude, linewidth=0.8, color='r', label='Amplitude')
axes[1].legend()
axes[1].set_xlim(0)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude (μV)')


axes[2].plot(fft_freq, power, linewidth=0.8, color='b')
axes[2].legend(['Power'])
axes[2].set_xlim(0)
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('Power (μV²)')


# psd 图
axes[3].plot(fft_freq, psd, linewidth=0.8, color='green')
axes[3].legend(['PSD'])
axes[3].set_xlim(0)
axes[3].set_xlabel('Frequency (Hz)')
axes[3].set_ylabel('PSD (μV²/Hz)')


plt.tight_layout()
plt.show()
